In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 4
:caption: Contents:

# Nonstationary case: walkthrough

```{warning}
The templates for the swan runs are planned to be updated in the near future to provide more flexibility to users who want to run cases under tailor-made conditions. The current templates assumes runs to be made under:

- Nautical conventions for wave direction
- Spherical coordinates
- Two-dimensional domains
- Specific numerics and physics (user is refered to the template files for details)
```

## Imports

In [2]:
import matplotlib.pyplot as plt
from datetime import datetime, timedelta


In [3]:
from oceanicospy.models.swanpy import Initializer
from oceanicospy.models.swanpy.preprocess import *
from oceanicospy.models.swanpy.execution import CaseRunner
from oceanicospy.models.swanpy.postprocess import SwanOutputReader
from oceanicospy.plots import *

## Workflow overview

This notebook walks through a complete SWAN **non-stationary** modelling workflow using the `swanpy` subpackage inside `oceanicospy`. The example is based on a non-stationary hindcast for the San Andres island in Colombia during May 2023. 

There is a parent domain that covers the entire island **TO_BE_COMPLETED**

Unlike the stationary case, a non-stationary run integrates the wave action equation over time, requiring:
* A simulation window (`ini_date` / `end_date`)
* Time-varying wind forcing (spatial ERA5 or CMDS fields)
* Time-varying water level forcing (tide gauge data)
* Time-varying boundary conditions

This scenario is also nested which means that the model is run in several domains with increasing resolution, where the output of the coarser domain is used as input for the finer domain. This allows to capture large scale wave generation processes while resolving local wave dynamics nearshore.

```{note}
The workflow assumes that input data (bathymetry `.bot`, friction `.fric`, and output points) are already placed in the appropriate `input/domain_0X/` folder inside the case directory. Tide gauge data and ERA5 wind and wave data are downloaded automatically if not already present.

## Case configuration

Four parameters drive the non-stationary setup:

| Parameter | Description |
|---|---|
| `path_case` | Root directory holding `input/`, `run/`, `output/`, `pros/` |
| `dict_ini_data` | Metadata dictionary consumed by `Initializer` |
| `ini_date` | Start of the simulation window in datetime format |
| `end_date` | End of the simulation window in datetime format |

The `dict_ini_data` dictionary accepts the following keys:

- `name`: A string identifier for the case.
- `case_number` (optional): Integer to differentiate between multiple cases.
- `case_description`: Brief description of the case.
- `stat_id`: An integer indicating the type of run (0 for non-stationary, 1 for stationary).
- `number_domains`: Total number of computational domains.
- `parent_domains`: Dict mapping each domain id to its parent domain id (or `None` for top-level domains). Required for `write_nest_section()` to correctly handle nesting output.

### Domain iteration workflow

All the definition of spatial information for each type of input will be done using an iteration over the domains, which is a common workflow for nested cases. This allows to easily scale up to any number of domains and correctly link parent-child relationships for nesting. First, we define a dictionary with the necessary information for each domain, and then we loop through it to initialize the case and populate the `run.swn` files.

In [4]:
domains = {
    1: {
        "grid": dict(lon_ll_corner=278.2, lat_ll_corner=12.4, x_extent=0.2, y_extent=0.3, nx=222, ny=333),
        "bathy": dict(lon_ll_corner_bot=278.1811, lat_ll_corner_bot=12.372, nx_bot=322, ny_bot=422, dx_bot=0.0009, dy_bot=0.0009),
        "fric": dict(lon_ll_corner_fric=278.1811, lat_ll_corner_fric=12.372, nx_fric=322, ny_fric=422, dx_fric=0.0009, dy_fric=0.0009),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=278.1811, lat_ll_corner_wl=12.372, nx_wl=322, ny_wl=422, dx_wl=0.0009, dy_wl=0.0009),
        "parent": None
    },
    2: {
        "grid": dict(lon_ll_corner=278.2265167, lat_ll_corner=12.4413166, x_extent=0.13185, y_extent=0.1908, nx=293, ny=424),
        "bathy": dict(lon_ll_corner_bot=278.2265167, lat_ll_corner_bot=12.4413166, nx_bot=293, ny_bot=424, dx_bot=0.00045, dy_bot=0.00045),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=278.2265167, lat_ll_corner_wl=12.4413166, nx_wl=293, ny_wl=424, dx_wl=0.00045, dy_wl=0.00045),
        "fric": dict(lon_ll_corner_fric=278.2265167, lat_ll_corner_fric=12.4413166, nx_fric=293, ny_fric=424, dx_fric=0.00045, dy_fric=0.00045),
        "parent": 1
    },
    3: {
        "grid": dict(lon_ll_corner=278.270165, lat_ll_corner=12.550217, x_extent=0.0747, y_extent=0.0747, nx=332, ny=332),
        "bathy": dict(lon_ll_corner_bot=278.270165, lat_ll_corner_bot=12.550217, nx_bot=332, ny_bot=332, dx_bot=0.000225, dy_bot=0.000225),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=278.270165, lat_ll_corner_wl=12.550217, nx_wl=332, ny_wl=332, dx_wl=0.000225, dy_wl=0.000225),
        "fric": dict(lon_ll_corner_fric=278.270165, lat_ll_corner_fric=12.550217, nx_fric=332, ny_fric=332, dx_fric=0.000225, dy_fric=0.000225),
        "parent": 2
    },
    4: {
        "grid": dict(lon_ll_corner=360 - 81.72, lat_ll_corner=12.5, x_extent=0.055, y_extent=0.055, nx=243, ny=243),
        "bathy": dict(lon_ll_corner_bot=360 - 81.72, lat_ll_corner_bot=12.5, nx_bot=243, ny_bot=243, dx_bot=0.000225, dy_bot=0.000225),
        "wind": dict(lon_ll_corner_wind=278, lat_ll_corner_wind=12.3, nx_wind=24, ny_wind=24, dx_wind=0.025, dy_wind=0.025),
        "wl":   dict(lon_ll_corner_wl=360 - 81.72, lat_ll_corner_wl=12.5, nx_wl=243, ny_wl=243, dx_wl=0.000225, dy_wl=0.000225),
        "fric": dict(lon_ll_corner_fric=360 - 81.72, lat_ll_corner_fric=12.5, nx_fric=243, ny_fric=243, dx_fric=0.000225, dy_fric=0.000225),
        "parent": 2
    }
}

Each key in the `domains` dictionary corresponds to a domain id, and the value is another dictionary with the following keys:
- `grid`: Dict with grid parameters (`lon_ll_corner`, `lat_ll_corner`, `x_extent`, `y_extent`, `nx`, `ny`).

The following dictionaries share the same structure: `lon_ll_corner_suffix`, `lat_ll_corner_suffix`, `nx_suffix`, `ny_suffix`, `dx_suffix`, `dy_suffix`. Where `suffix` is replaced by the type of input (e.g., `bathy`, `wind`, `wl`, `fric`)
- `bathy`: Dict with spatial bathymetry parameters
- `wind`: Dict with spatial wind parameters.
- `wl`: Dict with spatial water level parameters.
- `fric`: Dict with spatial friction parameters.

There is also another key that maps each domain id to its parent domain id:

- `parent`: Parent domain id (or `None` for top-level domains).


In [5]:
path_case = '../../data/model_runs/nonstationary_swan/'

ini_date = datetime(2023, 5, 7, 0)
end_date = datetime(2023, 5, 31, 23)

dict_ini_data = dict(
    name             = 'Non-stationary case for San Andres',
    case_number      = 2,
    case_description = 'Example non-stat',
    stat_id          = 0, # 0 → non-stationary
    number_domains   = len(domains),
    parent_domains   = {d:cfg["parent"] for d, cfg in domains.items()} # domain 1 has no parent and domain 3 and 4 share parent domain 2.
)

```{hint}
`stat_id` convention
* `stat_id = 0` → **non-stationary** run  ← _used in this notebook_
* `stat_id = 1` → **stationary** run

## Initialization

`Initializer` creates the project directory tree and writes the baseline SWAN configuration file (`run.swn`) from the **non-stationary template** (`run_base_nonstat.swn`). The directory layout is:

```
path_case/
├── input/
│   └── domain_01/          ← place .bot, .fric and output points here
│   └── domain_02/ 
│   └── domain_03/ 
│   └── domain_04/ 
├── pros/
├── run/
│   └── domain_01/
│       └── run.swn         ← generated from non-stationary template
│   └── domain_02/
│       └── run.swn
│   └── domain_03/
│       └── run.swn
│   └── domain_04/
│       └── run.swn
└── output/
    └── domain_01/
    └── domain_02/
    └── domain_03/
    └── domain_04/
```

```{note}
Pass `ini_date` and `end_date` to `Initializer` for non-stationary runs. These are stored in `case.ini_date` and `case.end_date` and used by modules like `WindForcing`, `WaterLevelForcing`, and `BoundaryConditions` to slice the downloaded time series to the simulation window.

In [6]:
case = Initializer(
    root_path     = path_case,
    dict_ini_data = dict_ini_data,
    ini_date      = ini_date,
    end_date      = end_date,
)

case.create_folders()       # creates all project folders
case.replace_ini_data()     # writes run.swn from the NONSTAT template

*** Initializing SWAN model ***


	*** Creating project folder structure ***


	*** Copying base SWAN configuration file into run folder ***



## Creating the grid

`GridMaker` populates the `CGRID` section of `run.swn` for every computational domain. We'll reach this using that method under an iteration over the domains.

```{info}
    `GridMaker` also has the option to create the grid from the bathymetry data.
```

In [8]:
for domain_id,config in domains.items():
    domain_grid_dict = config["grid"]
    case_grid = GridMaker(init = case,
                          domain_number = domain_id, 
                          grid_info = domain_grid_dict)
    case_grid.fill_grid_section()


*** Initializing gridmaker for domain 1 ***


 	*** Adding/Editing grid information for domain 1 in configuration file ***


*** Initializing gridmaker for domain 2 ***


 	*** Adding/Editing grid information for domain 2 in configuration file ***


*** Initializing gridmaker for domain 3 ***


 	*** Adding/Editing grid information for domain 3 in configuration file ***


*** Initializing gridmaker for domain 4 ***


 	*** Adding/Editing grid information for domain 4 in configuration file ***



## Setting up the bathymetry

`BathyMaker` fills the `BOTTOM` section of each `run.swn` for each domain. Use `use_ascii_file_from_user()` when a pre-processed `.bot` file already exists in `input/domain_XX/`.

| Method | When to use |
|---|---|
| `use_ascii_file_from_user` | when a pre-processed `.bot` file already exists in `input/domain_0X/`. This method implies the use of a dictionary with bathymetry metadata. |

```{note}
An iteration is made for every type of input data only for illustration purposes. In practice, a single iteration can be made where all the necessary methods are called for each domain, which is more efficient. The iteration is separated here for clarity.


In [9]:
for domain_id,config in domains.items():
    domain_bathy_dict = config["bathy"]
    case_bathy = BathyMaker(init = case,
                          domain_number = domain_id, 
                          bathy_info = domain_bathy_dict)
    case_bathy.use_ascii_file_from_user()
    case_bathy.fill_bathy_section()


*** Initializing bathymaker for domain 1 ***


 	*** Adding/Editing bathymetry information for domain 1 in configuration file ***


*** Initializing bathymaker for domain 2 ***


 	*** Adding/Editing bathymetry information for domain 2 in configuration file ***


*** Initializing bathymaker for domain 3 ***


 	*** Adding/Editing bathymetry information for domain 3 in configuration file ***


*** Initializing bathymaker for domain 4 ***


 	*** Adding/Editing bathymetry information for domain 4 in configuration file ***



## Defining bottom friction information

`BottomFrictionProcessor` is a class inside `swanpy` that fills the `FRICTION` section of `run.swn` with the appropriate parameters for bottom friction. Similarly to the bathymetry, there is one method available to set up the bottom friction file for the model based on existing information in the `input/domain_0X/` folder:

| Method | When to use |
|---|---|
| `use_ascii_file_from_user` | when a pre-processed `.fric` file already exists in `input/domain_0X/`. This method implies the use of a dictionary with bottom friction metadata. |

In [10]:
for domain_id,config in domains.items():
    domain_bathy_dict = config["fric"]
    case_bathy = BottomFrictionProcessor(init = case,
                          domain_number = domain_id, 
                          bottom_fric_info = domain_bathy_dict)
    case_bathy.use_ascii_file_from_user()
    case_bathy.fill_friction_section()


*** Initializing BottomFrictionProcessor for domain 1 ***


 	*** Adding/Editing friction information for domain 1 in configuration file ***


*** Initializing BottomFrictionProcessor for domain 2 ***


 	*** Adding/Editing friction information for domain 2 in configuration file ***


*** Initializing BottomFrictionProcessor for domain 3 ***


 	*** Adding/Editing friction information for domain 3 in configuration file ***


*** Initializing BottomFrictionProcessor for domain 4 ***


 	*** Adding/Editing friction information for domain 4 in configuration file ***



## Defining wind forcing from ERA5

For this non-stationary run, SWAN requires a **spatially distributed, time-varying** wind field rather than a single constant vector. `WindForcing` is designed to handle the following steps:

1. **Downloading** ERA5 (or CMDS) 10-m wind components for the simulation window.
2. **Converting** the NetCDF output to SWAN's binary-equivalent ASCII format.
3. **Writing** the `INP WIND` / `READ WIND` section of `run.swn`.

The spatial grid for the wind field is specified through `wind_info`. It does not need to match the computational grid resolution, but must cover the entire domain.

```{note}
When `share_winds=True` (the default), domain 1 downloads the wind data and subsequent domains reuse it, which is the recommended approach for nested multi-domain cases.

In [11]:
for domain_id,config in domains.items():
    domain_wind_dict = config["wind"]
    case_wind = WindForcing(init = case,
                          domain_number = domain_id, 
                          wind_info = domain_wind_dict)
    
    # Skips download if the file already exists in input/domain_01/
    case_wind.get_winds_from_ERA5(utc_offset_hours=-5)

    # Convert NetCDF → SWAN ASCII and link into run/domain_01/
    case_wind.write_ERA5_ascii('winds_era5.nc', 'winds.wnd')
    case_wind.fill_wind_section()


*** Initializing winds for domain 1 ***

the file ../../data/model_runs/nonstationary_swan/input/domain_01/winds_era5.nc already exists
	 ERA5 wind data already exists, skipping download


/Users/franklinayala/Documents/projects/oceanicospy/oceanicospy/models/swanpy/preprocess/wind_forcing.py:78: FutureWarning: DatetimeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  time_to_write = time.format(formatter=lambda x: x.strftime('%Y%m%d.%H%M'))


	 ERA5 wind data converted to ASCII format and saved as winds.wnd

 	*** Adding/Editing winds information for domain 1 in configuration file ***


*** Initializing winds for domain 2 ***

	 ERA5 wind data already exists in domain 1, skipping download
	 ERA5 wind data converted to ASCII format and saved as winds.wnd in domain 01, linking to domain 2

 	*** Adding/Editing winds information for domain 2 in configuration file ***


*** Initializing winds for domain 3 ***

	 ERA5 wind data already exists in domain 1, skipping download
	 ERA5 wind data converted to ASCII format and saved as winds.wnd in domain 01, linking to domain 3

 	*** Adding/Editing winds information for domain 3 in configuration file ***


*** Initializing winds for domain 4 ***

	 ERA5 wind data already exists in domain 1, skipping download
	 ERA5 wind data converted to ASCII format and saved as winds.wnd in domain 01, linking to domain 4

 	*** Adding/Editing winds information for domain 4 in configuration file ***


```{hint}
To use CMDS wind data instead of ERA5, replace `get_winds_from_ERA5` / `write_ERA5_ascii` with
`get_winds_from_CMDS` / `write_CMDS_ascii`. The rest of the workflow is identical.

## Setting up the water level forcing

`WaterLevelForcing` handles tide-gauge water level data from UHSLC (University of Hawaii Sea Level Center) through the following steps:

1. **Downloading** hourly tide-gauge data for the simulation window from UHSLC.
2. **Converting** the CSV to a spatially uniform SWAN ASCII water-level file (one grid snapshot per timestep).
3. **Writing** the `INP WLEV` / `READ WLEV` section of `run.swn`.

```{warning}
For this example, the spatial grid for the water level is written based on the bathymetry grid of the domain. **TO_BE_COMPLETED**
```

The San Andres UHSLC station code is **737** according to the [UHSLC station list](https://uhslc.soest.hawaii.edu/stations/).

```{hint}
It was found that the tide-gauge data from UHSLC has some missing changes on the reference datum between 1997 and 2018 so the records are shifted to match the correct datum. For this reason, the `get_waterlevel_from_UHSLC()` delivers a pd.DataFrame that can be easily manipulated by the user to apply the necessary corrections before writing the SWAN ASCII file.

The `write_UHSLC_ascii()` method has an optional argument `detrend_wl` that applies linear detrending to the water level data before writing. This can be useful to remove any long-term trends in the tide gauge data that may not be relevant for the simulation window. By default, `detrend_wl` is set to `True`, but users can set it to `False` if they prefer to keep the original water level data without detrending.

In [12]:
for domain_id,config in domains.items():
    domain_wl_dict = config["wl"]
    case_wl = WaterLevelForcing(init = case,
                          domain_number = domain_id, 
                          wl_info = domain_wl_dict)
    df_wl = case_wl.get_waterlevel_from_UHSLC(station_id=737)
    
    # mask application
    correction_mask = (
        (df_wl.index >= datetime(1997, 1, 1, 0)) &
        (df_wl.index <= datetime(2018, 12, 31, 18))
    )
    df_wl.loc[correction_mask, "depth[m]"] -= 2.0

    case_wl.write_UHSLC_ascii(df_wl, 'water_levels.wl')

    #Convert CSV → SWAN ASCII
    case_wl.fill_wl_section()


*** Initializing water levels for domain 1 ***

the file ../../data/model_runs/nonstationary_swan/input/domain_01/h737.csv already exists
	 UHSLC water level data already exists, skipping download
	 UHSLC water level data converted to ASCII format and saved as water_levels.wl

 	*** Adding/Editing water level information for domain 1 in configuration file ***


*** Initializing water levels for domain 2 ***

	 UHSLC water level data already exists in domain 1, skipping download
	 UHSLC water level data converted to ASCII format and saved as water_levels.wl in domain 01, linking to domain 2

 	*** Adding/Editing water level information for domain 2 in configuration file ***


*** Initializing water levels for domain 3 ***

	 UHSLC water level data already exists in domain 1, skipping download
	 UHSLC water level data converted to ASCII format and saved as water_levels.wl in domain 01, linking to domain 3

 	*** Adding/Editing water level information for domain 3 in configuration file *

## Defining boundary conditions

For non-stationary runs, `swanpy` supports two approaches to spectral boundary conditions in the sides of the computational grid:

| Approach | SWAN example command |
|---|---|
| **Constant** (`CON PAR`) | `BOUN SIDE N CLOCKW CON PAR Hs Tp Dir dd` |
| **Time-varying** (`VAR FILE`) | `BOUN SIDE N CLOCKW VAR FILE …` |

Using `'segment'` as `bound_type` in the `bound_info` dictionary is not supported.

For this scenario, the time-varying TPAR boundary conditions are downloaded from ERA5 (or CMDS) wave reanalysis at a set of boundary points. The workflow is:

1. `get_waves_from_ERA5()` – download ERA5 wave parameters (Hs, Tp, MWD) for the region.
2. `tpar_from_ERA5()` – extract point time series at the boundary corners and write individual `.bnd` TPAR files required by SWAN.
3. `create_boundary_line()` - After generating the TPAR files, add the corresponding `BOUN SIDE VAR FILE` commands to `run.swn` following SWAN syntax.

Because the sense of a nested simulation is that the coarser domain can transmit wave information to the finer domain, only the outermost domain (domain 1) needs to have time-varying boundary conditions.

The set of points where the boundary conditions are extracted needs to be defined using their longitude and latitude coordinates:

In [13]:
# Boundary point arrays — define the lat/lon grid of boundary sample points
lat_boundaries_dom01 = [12.40,12.50,12.60,12.70]
lon_boundaries_dom01 = [-81.80,-81.70,-81.60]

The `bound_info` dictionary is expected to have the following keys to define the boundary conditions for each domain:

- `bound_type`: A string with the type of boundary condition ("side" is supported)
- `variable_bound`: A boolean indicating whether the boundary condition is time-variable (True) or constant (False)

In [15]:
for domain_id,config in domains.items():
    domain_wind_dict = config["wind"]
    case_boundary = BoundaryConditions(
        init          = case,
        domain_number = domain_id,
        bound_info     = {'bound_type': 'side','variable_bound': True},
    )

    # Step 1 – download ERA5 wave parameters for the boundary region
    case_boundary.get_waves_from_ERA5(
        utc_offset_hours = -5,
        wind_info_dict    = domain_wind_dict)
    
    # Step 2 - extract time series at boundary corners and write .bnd TPAR files

    case_boundary.tpar_from_ERA5(
        points_lat = lat_boundaries_dom01,
        points_lon = lon_boundaries_dom01,
    )

    # Step 3 - create the BOUN lines for the .swn file
    case_boundary.create_boundary_line(list_sides=['N','S','E','W'], points_lon=lon_boundaries_dom01, points_lat=lat_boundaries_dom01)

    case_boundary.fill_boundaries_section()


*** Initializing boundary conditions for domain 1 ***

the file ../../data/model_runs/nonstationary_swan/input/domain_01/waves_era5.nc already exists
	 ERA5 wave data already exists, skipping download
	*** Finished processing boundary files for domain 1 ***

BOUN SIDE N CLOCKW VAR FILE 0.0 '../../data/model_runs/nonstationary_swan/input/domain_01/TparN_1.bnd' 1 & 
0.1 '../../data/model_runs/nonstationary_swan/input/domain_01/TparN_2.bnd' 1 & 
0.2 '../../data/model_runs/nonstationary_swan/input/domain_01/TparN_3.bnd' 1 &
BOUN SIDE S CLOCKW VAR FILE 0.0 '../../data/model_runs/nonstationary_swan/input/domain_01/TparS_1.bnd' 1 & 
0.1 '../../data/model_runs/nonstationary_swan/input/domain_01/TparS_2.bnd' 1 & 
0.2 '../../data/model_runs/nonstationary_swan/input/domain_01/TparS_3.bnd' 1 &
BOUN SIDE E CLOCKW VAR FILE 0.0 '../../data/model_runs/nonstationary_swan/input/domain_01/TparE_1.bnd' 1 & 
0.1 '../../data/model_runs/nonstationary_swan/input/domain_01/TparE_2.bnd' 1 & 
0.2 '../../data/mo

```{note}
After calling `tpar_from_ERA5()`, the `.bnd` files are generated in `input/domain_01/` . To activate them in SWAN, the corresponding `BOUN SIDE VAR FILE` lines need to be added to `run.swn`

## Computation and run configuration

Finally, after all the input data is set up, the `CaseRunner` class handles the final steps to finish the configuration of the case and prepare it for running. For a nested non-stationary run the following steps are performed:

* `define_output_from_file()` – reads a CSV of output point coordinates for each domain and writes `points.loc` file.
* `write_nest_section()` – fills the nest section of each `run.swn` with the appropriate `NESTOUT`/`NGRID` commands for the nesting configuration of the case. 
* `fill_computation_section()` – writes the required line `COMP NONSTAT ini_date dt_min MIN end_date` into `run.swn`.

Since each domain can have different grid resolutions and therefore different time step requirements, the `fill_computation_section()` method is called separately for each domain with a domain-specific `dict_comp_data`. The `dt_min` key can be adjusted for each domain to ensure numerical stability while optimizing computation time. Other parameters can be kept the same across domains if the simulation window and output requirements are the same. 

All the keys in `dict_comp_data` are described as follows:

| Key | Description |
|---|---|
| `stat_comp` | `1` if the computation is stationary, `0` if it is non-stationary |
| `ini_comp_date` | Start date of the computation (datetime format) |
| `dt_min` | Time step in minutes |
| `end_comp_date` | End date of the computation (datetime format) |
| `ini_output_date` | First output dump date (allows a spin-up period before saving) in datetime format |
| `output_res_min` | Output interval in minutes |
| `ini_nest_date` | Date of the first nest output (for multi-domain cases) in datetime format |

The common configuration for all domains is stored in a dictionary called `_common` that is added as a base to the domain-specific `dict_comp_data` when calling `fill_computation_section()`. This allows to easily maintain the common parameters while adjusting domain-specific ones.

In [17]:
_common = dict(                                                                                                                   
    ini_output_date = (ini_date + timedelta(days=5)),                                                     
    output_res_min  = 60,
    stat_comp       = 0,                                                                                                            
    dt_min          = 10,                                                                                                         
    end_comp_date   = end_date,                                                                           
)

compilation_data = {
    1: {**_common, "ini_comp_date": ini_date,                                                             
                    "ini_nest_date": ini_date + timedelta(days=2)},                                      
    2: {**_common, "ini_comp_date": ini_date + timedelta(days=2),                                       
                    "ini_nest_date": ini_date + timedelta(days=2)},                                      
    3: {**_common, "ini_comp_date": ini_date + timedelta(days=2)},                                      
    4: {**_common, "ini_comp_date": ini_date + timedelta(days=2)},                                      
}  

Each domain has a file with the output points defined in `input/domain_0X/SalidasSWAN.txt`.

In [ ]:
for domain_id, config in domains.items():
    case_compilation = CaseRunner(
        init          = case,
        domain_number = domain_id,
        dict_comp_data= compilation_data[domain_id],
        all_domains   = domains
    )

    # Write output point locations from CSV (X, Y columns)
    case_compilation.define_output_from_file(filename='SalidasSWAN.txt')

    # Remove NESTOUT/NGRID template lines (single-domain has no child domains)
    case_compilation.write_nest_section()

    # Write the COMP command and output date/interval placeholders
    case_compilation.fill_computation_section()

All configuration is now written to `run/domain_0X/run.swn`. The case and its domains are ready to be executed with the SWAN binary locally or submitted to an HPC scheduler.

## Reading the output

After running the non-stationary case, the output files will be generated in the `output/domain_0X/` folders and can be post-processed using the `SwanOutputReader` class in `swanpy`. This class provides methods to read and extract information from the SWAN output files, such as time-varying wave parameters at output points, directional wave spectra, and spatial maps across the entire grid. Because the run is non-stationary, all outputs carry a `time` dimension that spans the simulation window.

In [ ]:
out_reader = SwanOutputReader()

# Domain to post-process — change to explore other domains
DOMAIN_ID = 4
output_dir = f'../../data/model_runs/nonstationary_swan/output/domain_{DOMAIN_ID:02d}/'

### Point output

The output points defined in the `.loc` file can be read using the `read_point_output()` method. For a non-stationary run the result is a DataFrame with a multi-level index `(time, point)`, where every row corresponds to one output point at one simulation timestep. This allows you to extract and plot the wave parameter time series at each location.

Two parameters are required:
- `output_dir`: directory where the output files are located
- `filename`: name of the point-output file (e.g. `PointSWAN.out`)

In [ ]:
point_df = out_reader.read_point_output(
    output_dir = output_dir,
    filename   = 'PointSWAN.out'
)

point_df

A quick plot of the significant wave height (`Hsig`) time series at the first output point illustrates the time-varying nature of the non-stationary results.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for pt, grp in point_df.groupby(level='point'):
    ax.plot(grp.index.get_level_values('time'), grp['Hsig'], label=f'Point {pt}')
ax.set(xlabel='Date', ylabel='Hsig (m)', title=f'Significant wave height — domain {DOMAIN_ID:02d}')
ax.legend()
plt.tight_layout()
plt.show()

### Wave spectra

The directional wave spectra output is read with `read_spectral_output()`. For a non-stationary run the returned `xarray.Dataset` has dimensions `(time, site, freq, dir)`, where `time` spans the entire simulation window. The method requires the same `output_dir` and `filename` arguments as the point-output reader.

In [ ]:
wave_spectra = out_reader.read_spectral_output(
    output_dir = output_dir,
    filename   = 'SpecSWAN.out'
)

wave_spectra

In [ ]:
wave_spectra.coords

A polar plot of the directional variance density for a selected time step gives a compact overview of the dominant wave energy distribution in frequency–direction space.

In [ ]:
# Select the time step to visualise (−1 = last snapshot)
t_idx = -1

theta, r = np.meshgrid(wave_spectra.dir.values, wave_spectra.freq.values)
fig, ax = plt.subplots(figsize=(5, 4), subplot_kw={'projection': 'polar'})
cf = ax.contourf(np.deg2rad(theta), r, wave_spectra['efth'][t_idx, 0, :, :],
                  levels=20, cmap='afmhot_r')
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set(ylim=(0, 0.3))
plt.colorbar(cf, label='Energy Density ($m^2/Hz$)', pad=0.1)
ax.set_title(f"Wave spectra — {str(wave_spectra.time.values[t_idx])[:16]}")
plt.show()

In [ ]:
wave_spectra['efth'][t_idx, 0, :, :]

### Spatial output

Spatial maps of wave parameters are read with `read_spatial_output()`. For a non-stationary case the dataset includes a `time` dimension, so each variable (e.g. `Hsig`) has shape `(time, lat, lon)`. The method can infer spatial dimensions automatically or use the `grid_info` dictionary that was passed to `GridMaker` during case configuration. Passing `grid_info` is recommended to ensure correct coordinate labelling.

In [ ]:
grid_info = domains[DOMAIN_ID]['grid']

wave_field_ds = out_reader.read_spatial_output(
    output_dir = output_dir,
    filename   = 'wave_field.mat',
    grid_info  = grid_info
)

wave_field_ds

A quick map of the significant wave height at the selected snapshot, together with the output point locations, illustrates the spatial wave field produced by the non-stationary simulation.

In [ ]:
# Reuse t_idx from the spectra cell; change as needed
unique_pts = point_df[['Xp', 'Yp']].drop_duplicates()

fig, ax = plt.subplots(figsize=(3, 4))
cf = ax.contourf(wave_field_ds.lon, wave_field_ds.lat,
                  wave_field_ds['Hsig'].isel(time=t_idx), levels=20)
ax.plot(unique_pts['Xp'], unique_pts['Yp'], 'ro', label='Output points')
ax.set(xlabel='Longitude', ylabel='Latitude',
       title=f"Hsig — {str(wave_field_ds.time.values[t_idx])[:16]}")
plt.colorbar(cf, label='Significant Wave Height (m)', fraction=0.08)
plt.show()